# Phase 13 — CTGAN Behavioral Simulator: Training Notebook

Trains a `CTGANSynthesizer` (SDV) on `analytics.session_features` enriched with
a per-session primary category derived from `retailrocket_raw.item_properties`.

**Outputs**
- `data/session_features.parquet` — training corpus (gitignored)
- `models/ctgan_sessions.pkl` — trained synthesizer (gitignored)
- `docs/ctgan_evaluation.md` — JS divergence summary + evaluation plots

**Runtime**: ~30–90 min on CPU for 300 epochs with ~1M Retailrocket sessions.
Run with `make ctgan-train` which executes this notebook via `nbconvert`.

In [ ]:
# Cell 1 — Imports and ClickHouse connection
import os
import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 4)

import clickhouse_connect

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger(__name__)

# Resolve repo root so src/ imports work from notebooks/
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

CH = clickhouse_connect.get_client(
    host=os.getenv('CLICKHOUSE_HOST', 'localhost'),
    port=int(os.getenv('CLICKHOUSE_PORT', '8123')),
    username=os.getenv('CLICKHOUSE_USER', 'analytics'),
    password=os.getenv('CLICKHOUSE_PASSWORD', 'analytics_password'),
)

# Verify connectivity
result = CH.query('SELECT count() FROM analytics.session_features WHERE source = \'retailrocket\'')
N_REAL = int(result.result_rows[0][0])
log.info('Retailrocket sessions available for training: %s', f'{N_REAL:,}')
assert N_REAL > 0, 'No Retailrocket data found — run make retailrocket-import first.'

In [ ]:
# Cell 2 — Export session_features + primary_category to Parquet
#
# We export Retailrocket sessions only (the ML training corpus — see Phase 11).
# Live sessions are small and lack scroll/search coverage; adding them risks noise.
#
# Category derivation:
#   1. For each session, find the most-viewed item (argMax over view count).
#   2. Look up that item's category from item_properties (property = 'categoryid').
#   3. Rows with no category get '0' (unknown).
#
# score_tier is derived here (not joined from lead_scores_rule_based) so the
# training data does not require the scoring view to be populated first.

TRAINING_QUERY = """
WITH
    -- Most recent category per item from Retailrocket item properties
    item_cats AS (
        SELECT
            item_id,
            toString(toUInt32OrZero(argMax(value, event_time))) AS category_id
        FROM retailrocket_raw.item_properties
        WHERE property = 'categoryid'
        GROUP BY item_id
    ),
    -- Primary (most-viewed) item per synthesised session key
    session_primary_item AS (
        SELECT
            concat(toString(visitor_id), '_', toString(event_date)) AS session_id,
            argMax(item_id, view_cnt) AS primary_item_id
        FROM (
            SELECT
                visitor_id,
                item_id,
                toDate(event_time) AS event_date,
                countIf(event_type = 'view') AS view_cnt
            FROM retailrocket_raw.events
            GROUP BY visitor_id, item_id, event_date
        )
        GROUP BY session_id
    )
SELECT
    s.session_id AS session_id,
    s.page_views,
    s.product_views,
    s.add_to_cart_count,
    s.purchase_count,
    s.search_count,
    s.max_scroll_pct,
    s.session_duration_seconds,
    s.distinct_products_viewed,
    s.cart_abandoned,
    coalesce(ic.category_id, '0') AS primary_category,
    -- Derive score_tier from session signals (same weights as Phase 10 rules.py)
    multiIf(
        (
            if(s.add_to_cart_count > 0, 30, 0) +
            if(s.purchase_count > 0, 20, 0) +
            if(s.product_views >= 3, 15, 0) +
            if(s.search_count > 0, 10, 0) +
            if(s.max_scroll_pct > 70, 10, 0) +
            if(s.page_views = 0 AND s.add_to_cart_count = 0 AND s.purchase_count = 0 AND s.search_count = 0, -10, 0)
        ) >= 60, 'hot',
        (
            if(s.add_to_cart_count > 0, 30, 0) +
            if(s.purchase_count > 0, 20, 0) +
            if(s.product_views >= 3, 15, 0) +
            if(s.search_count > 0, 10, 0) +
            if(s.max_scroll_pct > 70, 10, 0) +
            if(s.page_views = 0 AND s.add_to_cart_count = 0 AND s.purchase_count = 0 AND s.search_count = 0, -10, 0)
        ) >= 30, 'warm',
        'cold'
    ) AS score_tier
FROM analytics.retailrocket_session_features s
LEFT JOIN session_primary_item spi ON s.session_id = spi.session_id
LEFT JOIN item_cats ic ON spi.primary_item_id = ic.item_id
"""

PARQUET_PATH = REPO_ROOT / 'data' / 'session_features.parquet'
PARQUET_PATH.parent.mkdir(parents=True, exist_ok=True)

log.info('Exporting training data from ClickHouse …')

result = CH.query(TRAINING_QUERY)
df_train = pd.DataFrame(result.result_rows, columns=result.column_names)

# Ensure correct types
df_train['max_scroll_pct'] = pd.to_numeric(df_train['max_scroll_pct'], errors='coerce').astype('float32')
df_train['session_duration_seconds'] = df_train['session_duration_seconds'].astype('int32')
df_train['cart_abandoned'] = df_train['cart_abandoned'].astype('int8')
for c in ['page_views', 'product_views', 'add_to_cart_count', 'purchase_count',
          'search_count', 'distinct_products_viewed']:
    df_train[c] = df_train[c].astype('int64')
df_train['primary_category'] = df_train['primary_category'].astype(str)
df_train['score_tier'] = df_train['score_tier'].astype(str)

# Drop session_id — not a feature; just a key used for deduplication above
df_train = df_train.drop(columns=['session_id'], errors='ignore')

# Cap primary_category cardinality to top-50 to prevent CTGAN OHE memory explosion.
# 1,079 raw categories would require ~13 GiB for one-hot encoding on 1.6M rows.
# Long-tail categories are merged into '0' (unknown); top-50 covers >80% of sessions.
TOP_N_CATS = 50
top_cats = df_train['primary_category'].value_counts().nlargest(TOP_N_CATS).index
df_train['primary_category'] = df_train['primary_category'].where(
    df_train['primary_category'].isin(top_cats), other='0'
)
log.info(
    'primary_category capped to top-%d (+ unknown). Distinct values now: %d',
    TOP_N_CATS, df_train['primary_category'].nunique(),
)

df_train.to_parquet(PARQUET_PATH, index=False)
log.info('Exported %d rows to %s', len(df_train), PARQUET_PATH)

print(f'Training corpus: {len(df_train):,} rows, {len(df_train.columns)} features')
print(df_train.dtypes)
print()
print('Class distribution (score_tier):')
print(df_train['score_tier'].value_counts(normalize=True).map('{:.2%}'.format))
print()
print('Top primary_category distribution (top 10):')
print(df_train['primary_category'].value_counts().head(10))

In [ ]:
# Cell 3 — Define SDV Metadata and train CTGANSynthesizer
#
# Continuous columns: numeric features with a natural real-valued domain.
# Categorical columns: discrete labels and flags.
#
# Training: 100 epochs on a stratified 200K-row sample.
# Full 1.6M-row corpus takes >12h on CPU; 200K rows preserves the statistical
# distribution (score_tier-stratified) and trains in ~30 min.

import joblib
from sdv.metadata import SingleTableMetadata
from sdv.single_table import CTGANSynthesizer

MODELS_DIR = REPO_ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)

EPOCHS = 100
BATCH_SIZE = 500
SAMPLE_SIZE = 200_000  # stratified sample; full corpus is ~1.6M rows

df_full = pd.read_parquet(PARQUET_PATH)

# Stratified sample by score_tier to preserve class balance
df_for_training = (
    df_full
    .groupby('score_tier', group_keys=False)
    .apply(lambda g: g.sample(
        n=min(len(g), int(SAMPLE_SIZE * len(g) / len(df_full))),
        random_state=42,
    ))
    .sample(frac=1, random_state=42)  # shuffle after stratification
    .reset_index(drop=True)
)
log.info(
    'Training on stratified sample: %d rows (%.1f%% of full corpus)',
    len(df_for_training), 100 * len(df_for_training) / len(df_full),
)
print('Sample score_tier distribution:')
print(df_for_training['score_tier'].value_counts(normalize=True).map('{:.2%}'.format))

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(df_for_training)

NUMERICAL_COLS = [
    'page_views', 'product_views', 'add_to_cart_count', 'purchase_count',
    'search_count', 'max_scroll_pct', 'session_duration_seconds',
    'distinct_products_viewed',
]
CATEGORICAL_COLS = ['cart_abandoned', 'score_tier', 'primary_category']

for col in NUMERICAL_COLS:
    metadata.update_column(col, sdtype='numerical')
for col in CATEGORICAL_COLS:
    metadata.update_column(col, sdtype='categorical')

log.info('Training CTGANSynthesizer for %d epochs (batch_size=%d) …', EPOCHS, BATCH_SIZE)

synthesizer = CTGANSynthesizer(
    metadata,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=True,
)
synthesizer.fit(df_for_training)

final_path = MODELS_DIR / 'ctgan_sessions.pkl'
joblib.dump(synthesizer, final_path)
log.info('Model saved → %s', final_path)

In [ ]:
# Cell 4 — Evaluate: marginal distributions and Jensen-Shannon divergence
#
# For each continuous feature, compare the real vs synthetic marginal distribution
# with a KDE plot and compute JSD (0 = identical, 1 = maximally different).
# Target: JSD < 0.10 per continuous feature (Phase 13 acceptance criterion).

from scipy.spatial.distance import jensenshannon
from scipy.stats import gaussian_kde

N_EVAL = min(5_000, len(df_for_training))
log.info('Generating %d synthetic rows for evaluation …', N_EVAL)
df_synthetic_eval = synthesizer.sample(num_rows=N_EVAL)

def compute_jsd_continuous(real: pd.Series, synthetic: pd.Series, n_points: int = 200) -> float:
    real_clean = real.dropna().to_numpy(dtype=float)
    synth_clean = synthetic.dropna().to_numpy(dtype=float)
    if len(real_clean) < 5 or len(synth_clean) < 5:
        return float('nan')
    lo = min(real_clean.min(), synth_clean.min())
    hi = max(real_clean.max(), synth_clean.max())
    if lo >= hi:
        return 0.0
    grid = np.linspace(lo, hi, n_points)
    p = gaussian_kde(real_clean)(grid) + 1e-10
    q = gaussian_kde(synth_clean)(grid) + 1e-10
    return float(jensenshannon(p / p.sum(), q / q.sum()))


def compute_jsd_categorical(real: pd.Series, synthetic: pd.Series) -> float:
    cats = set(real.dropna().unique()) | set(synthetic.dropna().unique())
    if not cats:
        return float('nan')
    real_freq = real.value_counts(normalize=True).reindex(cats, fill_value=0)
    synth_freq = synthetic.value_counts(normalize=True).reindex(cats, fill_value=0)
    p = (real_freq.values + 1e-10)
    q = (synth_freq.values + 1e-10)
    return float(jensenshannon(p / p.sum(), q / q.sum()))


jsd_results = {}

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

for i, col in enumerate(NUMERICAL_COLS):
    if col not in df_synthetic_eval.columns:
        continue
    jsd = compute_jsd_continuous(df_for_training[col], df_synthetic_eval[col])
    jsd_results[col] = jsd

    ax = axes[i]
    real_vals = df_for_training[col].dropna()
    synth_vals = df_synthetic_eval[col].dropna()
    clip_hi = real_vals.quantile(0.99)
    ax.hist(real_vals.clip(upper=clip_hi), bins=40, alpha=0.5, density=True, label='real', color='steelblue')
    ax.hist(synth_vals.clip(upper=clip_hi), bins=40, alpha=0.5, density=True, label='synthetic', color='darkorange')
    ax.set_title(f'{col}\nJSD={jsd:.4f}', fontsize=9)
    ax.legend(fontsize=7)
    ax.tick_params(labelsize=7)

for col in CATEGORICAL_COLS:
    if col not in df_synthetic_eval.columns:
        continue
    jsd = compute_jsd_categorical(df_for_training[col], df_synthetic_eval[col])
    jsd_results[col] = jsd

plt.tight_layout()
plot_path = REPO_ROOT / 'docs' / 'ctgan_kde_plots.png'
plot_path.parent.mkdir(exist_ok=True)
plt.savefig(plot_path, dpi=100)
plt.show()
log.info('KDE plots saved → %s', plot_path)

print('\n=== Jensen-Shannon Divergence per Feature ===')
print(f'{"Feature":<30} {"JSD":>8}  {"Status"}')
print('-' * 52)
THRESHOLD = 0.10
all_pass = True
for feat, jsd in sorted(jsd_results.items(), key=lambda x: -x[1]):
    status = '✓ OK' if jsd < THRESHOLD else '✗ ABOVE THRESHOLD'
    if jsd >= THRESHOLD:
        all_pass = False
    print(f'{feat:<30} {jsd:>8.4f}  {status}')

print()
if all_pass:
    print('✓ All features pass JSD < 0.10 threshold.')
else:
    print('⚠ Some features exceed JSD threshold. Review plots above.')

In [ ]:
# Cell 5 — Verify conditional sampling (Phase 17 prerequisite)
#
# Phase 17 requires CTGAN to condition on primary_category.
# This cell verifies that sample_from_conditions works for the top-5 categories
# and that the sampled rows respect the condition.

from sdv.sampling import Condition

top_categories = df_for_training['primary_category'].value_counts().head(5).index.tolist()
log.info('Testing conditional sampling for categories: %s', top_categories)

conditional_ok = True
for cat in top_categories:
    try:
        cond = Condition(num_rows=100, column_values={'primary_category': cat})
        sample = synthesizer.sample_from_conditions(conditions=[cond])
        cat_match = (sample['primary_category'] == cat).mean()
        status = '✓' if len(sample) > 0 else '✗ empty'
        print(f'  Category {cat:>6}: {len(sample):>4} rows, category_match={cat_match:.0%}  {status}')
        if len(sample) == 0:
            conditional_ok = False
    except Exception as exc:
        print(f'  Category {cat:>6}: FAILED — {exc}')
        conditional_ok = False

if conditional_ok:
    print('\n✓ Conditional sampling works for all tested categories.')
    print('  Phase 17 product_predictor.predict_for_product() can use sample_from_conditions().')
else:
    print('\n⚠ Conditional sampling failed for some categories.')
    print('  Phase 17 will fall back to unconditional sampling — expected for rare categories.')

In [ ]:
# Cell 6 — Quick smoke-test the generation script
#
# Runs generate_synthetic_sessions.py with --n-sessions 200 --dry-run to verify
# the full pipeline (load model → sample → score → format) without hitting ClickHouse.

import subprocess
import sys as _sys

_venv_bin = 'Scripts' if _sys.platform == 'win32' else 'bin'
venv_python = str(REPO_ROOT / '.venv-synth' / _venv_bin / 'python')
generate_script = str(REPO_ROOT / 'scripts' / 'generate_synthetic_sessions.py')

result = subprocess.run(
    [venv_python, generate_script, '--n-sessions', '200', '--dry-run'],
    capture_output=True,
    text=True,
    cwd=str(REPO_ROOT),
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError(f'generate_synthetic_sessions.py smoke test failed (exit {result.returncode})')
print('✓ Dry-run smoke test passed.')

In [ ]:
# Cell 7 — Write evaluation summary to docs/ctgan_evaluation.md

from datetime import datetime

lines = [
    '# CTGAN Evaluation Summary',
    '',
    f'**Generated:** {datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC")}',
    f'**Training corpus:** {len(df_for_training):,} Retailrocket sessions',
    f'**Epochs:** {EPOCHS}',
    f'**Batch size:** {BATCH_SIZE}',
    f'**Evaluation sample:** {N_EVAL:,} synthetic rows',
    f'**JSD threshold:** < {THRESHOLD}',
    '',
    '## Jensen-Shannon Divergence per Feature',
    '',
    '| Feature | JSD | Status |',
    '|---------|-----|--------|',
]
for feat, jsd in sorted(jsd_results.items(), key=lambda x: -x[1]):
    status = 'PASS' if jsd < THRESHOLD else 'FAIL'
    lines.append(f'| `{feat}` | {jsd:.4f} | {status} |')

lines += [
    '',
    '## Conditional Sampling',
    '',
    f'Tested top-5 categories. All returned rows: {"YES" if conditional_ok else "NO (some failed)"}.',
    '',
    '## Files',
    '',
    '- `models/ctgan_sessions.pkl` — trained CTGANSynthesizer (gitignored)',
    '- `data/session_features.parquet` — training corpus (gitignored)',
    '- `docs/ctgan_kde_plots.png` — marginal distribution plots',
    '',
    '## Regeneration',
    '',
    '```bash',
    'make synth-setup',
    'make ctgan-train',
    '```',
]

eval_path = REPO_ROOT / 'docs' / 'ctgan_evaluation.md'
eval_path.write_text('\n'.join(lines), encoding='utf-8')
log.info('Evaluation summary written → %s', eval_path)
print('\n'.join(lines))